In [2]:
import stim
import pymatching
import numpy as np
from itertools import combinations

QEC_cycle = 5

circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_x",
    before_measure_flip_probability=0.01,
    distance=3,
    rounds=QEC_cycle,
   # before_round_data_depolarization=p
)


dem = circuit.detector_error_model(decompose_errors=True)
matching = pymatching.Matching.from_detector_error_model(dem)

sampler = circuit.compile_detector_sampler()

num_det = dem.num_detectors


MAX_SYNDROME_WEIGHT = 2  # feasible for d=3

print(f"For QEC Round = {QEC_cycle} and Max weight = {MAX_SYNDROME_WEIGHT}")
print(f"Number of detectors = {num_det} ({QEC_cycle}*8, 8 detectors per cycle)")

lut = {}

for w in range(MAX_SYNDROME_WEIGHT + 1):
    for locs in combinations(range(num_det), w):
        syndrome = np.zeros(num_det, dtype=np.uint8)
        for i in locs:
            syndrome[i] = 1

        correction = matching.decode(syndrome, algorithm="mwpm")
        lut[tuple(syndrome.tolist())] = correction

print(f"LUT entries stored: {len(lut)}")

shots = 1000

detectors, observables = sampler.sample(
    shots=shots,
    separate_observables=True,
)

zero_correction = np.zeros_like(next(iter(lut.values())))

logical_errors = 0
missing_count = 0

for d, o in zip(detectors, observables):
    key = tuple(d.tolist())

    if key in lut:
        pred = lut[key]
    else:
        pred = zero_correction
        missing_count += 1

    if pred[0] != o[0]:
        logical_errors += 1

print("Logical error rate (pure LUT):", logical_errors / shots * 100)
print("Percent of missing (no correction) syndromes:", missing_count / shots*100)

For QEC Round = 5 and Max weight = 2
Number of detectors = 40 (5*8, 8 detectors per cycle)
LUT entries stored: 821
Logical error rate (pure LUT): 0.6
Percent of missing (no correction) syndromes: 7.3
